# NOORIX-AI Urdu Printed + Handwritten Line OCR — CRNN/CTC
**Prepared by Muhammad Farhan**

This is an Urdu-specific line-recognition starter. Run it separately for printed Urdu or handwritten Urdu by attaching one dataset at a time. It learns Unicode Urdu characters from your labelled line images and evaluates Character Error Rate (CER). It is not a page-layout/text-detection model; input images must already be cropped to individual text lines.

## Dataset to add
- **Printed Urdu:** search/add `UPTI 1.0 Urdu Printed Text Image Database` or use the **IIIT-Urdu OCR Dataset** (line images and transcriptions).
- **Handwritten Urdu:** use **NUST-UHWR** (line images with Unicode ground-truth transcriptions; official split files are available).
- Kaggle dataset search terms: `Urdu OCR line images transcription`, `Urdu handwriting recognition dataset`, `NUST UHWR Urdu handwriting`.

## Kaggle notebook title
`NOORIX-Urdu-Printed-Handwritten-CRNN-CTC-Muhammad-Farhan`

## Required manifest
Create a UTF-8 CSV with columns `image_path,text`, one line-image per row. For NUST-UHWR, convert the official tab-separated train/val/test manifest to CSV while preserving Urdu text. Do not train from page images without line crops and exact transcriptions. Keep an independent test set aside.


In [ ]:
!pip -q install jiwer
import os, json, glob, math, shutil, random
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for p in glob.glob("/kaggle/input/*"): print(p)


In [ ]:
# Select a CSV manifest for ONE dataset and ONE script type per run.
CSV_PATH="" # e.g. "/kaggle/input/nust-uhwr/metadata.csv"
IMAGE_COL="image_path"; TEXT_COL="text"
if not CSV_PATH: raise ValueError("Set CSV_PATH to your UTF-8 line-image transcription CSV.")
df=pd.read_csv(CSV_PATH)
df=df[[IMAGE_COL,TEXT_COL]].dropna().copy()
df.columns=["image_path","text"]
def resolve(p):
    p=Path(str(p))
    if p.is_file(): return str(p)
    for root in Path("/kaggle/input").glob("*"):
        q=root/p
        if q.is_file(): return str(q)
        hit=next(root.rglob(p.name),None)
        if hit: return str(hit)
    return None
df["image_path"]=df.image_path.apply(resolve)
df["text"]=df.text.astype(str)
df=df.dropna(subset=["image_path"])
df=df[df.text.str.len()>0].reset_index(drop=True)
print("Usable line pairs:",len(df)); display(df.head())
if len(df)<100: raise ValueError("Need at least 100 correctly labelled line images for a meaningful first experiment.")


In [ ]:
from sklearn.model_selection import train_test_split
train_df,val_df=train_test_split(df,test_size=0.1,random_state=42)
# Character vocabulary is learned only from training transcripts.
chars=sorted(set("".join(train_df.text.tolist())))
char_to_id={c:i+1 for i,c in enumerate(chars)} # 0 is CTC blank
id_to_char={i:c for c,i in char_to_id.items()}
NUM_CLASSES=len(chars)+1
print("Unique characters:",len(chars))


In [ ]:
HEIGHT=64
MAX_WIDTH=1600
def load_line(path):
    im=Image.open(path).convert("L")
    w=max(4,int(round(im.width*HEIGHT/im.height)))
    w=min(w,MAX_WIDTH)
    w=max(4,(w//4)*4)
    im=im.resize((w,HEIGHT),Image.Resampling.LANCZOS)
    arr=np.asarray(im,dtype=np.float32)/255.0
    return torch.from_numpy(arr).unsqueeze(0),w
class LineDataset(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        image,w=load_line(row.image_path)
        target=[char_to_id[c] for c in row.text if c in char_to_id]
        return image,torch.tensor(target,dtype=torch.long),w,row.text
def collate(batch):
    images,targets,widths,texts=zip(*batch)
    maxw=max(widths)
    padded=torch.zeros(len(images),1,HEIGHT,maxw)
    for i,x in enumerate(images): padded[i,:,:,:x.shape[-1]]=x
    target_lengths=torch.tensor([len(x) for x in targets],dtype=torch.long)
    flat=torch.cat(targets) if target_lengths.sum()>0 else torch.empty(0,dtype=torch.long)
    return padded,flat,target_lengths,torch.tensor([w//4 for w in widths]),list(texts)
train_ds=LineDataset(train_df); val_ds=LineDataset(val_df)
train_loader=DataLoader(train_ds,batch_size=8,shuffle=True,num_workers=2,collate_fn=collate)
val_loader=DataLoader(val_ds,batch_size=8,shuffle=False,num_workers=2,collate_fn=collate)


In [ ]:
class CRNN(nn.Module):
    def __init__(self,nclass):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU()
        )
        self.proj=nn.Linear(128*(HEIGHT//4),256)
        self.rnn=nn.LSTM(256,192,bidirectional=True,batch_first=True)
        self.head=nn.Linear(384,nclass)
    def forward(self,x):
        z=self.cnn(x)              # B,C,H,W/4
        z=z.permute(0,3,1,2).flatten(2)
        z=self.proj(z)
        z,_=self.rnn(z)
        return self.head(z).log_softmax(2)
device="cuda" if torch.cuda.is_available() else "cpu"
model=CRNN(NUM_CLASSES).to(device)
criterion=nn.CTCLoss(blank=0,zero_infinity=True)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)


In [ ]:
def greedy_decode(log_probs):
    ids=log_probs.argmax(2).tolist()
    result=[]
    for seq in ids:
        out=[]; prev=0
        for idx in seq:
            if idx!=0 and idx!=prev: out.append(id_to_char.get(idx,""))
            prev=idx
        result.append("".join(out))
    return result
def cer_one(ref,hyp):
    # Character error rate = Levenshtein distance / reference length.
    a=list(ref); b=list(hyp); prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1):
            cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]/max(1,len(a))


In [ ]:
# Initial baseline training. If GPU memory is limited, reduce batch_size above to 4.
EPOCHS=10
best_cer=999.0
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for images,targets,tgt_lens,in_lens,texts in train_loader:
        images=images.to(device); targets=targets.to(device)
        optimizer.zero_grad()
        logp=model(images).permute(1,0,2)
        loss=criterion(logp,targets,in_lens.to(device),tgt_lens.to(device))
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()
        losses.append(float(loss.item()))
    model.eval(); refs=[]; hyps=[]
    with torch.no_grad():
        for images,targets,tgt_lens,in_lens,texts in val_loader:
            pred=greedy_decode(model(images.to(device)))
            refs.extend(texts); hyps.extend(pred)
    val_cer=float(np.mean([cer_one(r,h) for r,h in zip(refs,hyps)]))
    print(f"Epoch {epoch+1}/{EPOCHS} | loss={np.mean(losses):.4f} | val_CER={val_cer:.4f}")
    if val_cer<best_cer:
        best_cer=val_cer
        torch.save({"state_dict":model.state_dict(),"chars":chars,"height":HEIGHT}, "/kaggle/working/noorix_urdu_crnn_best.pth")
print("Best validation CER:",best_cer)


In [ ]:
# Save the Urdu recognition model, Unicode vocabulary, metadata and ZIP.
out=Path("/kaggle/working/NOORIX_Urdu_Line_OCR_Muhammad_Farhan"); out.mkdir(exist_ok=True)
ckpt=torch.load("/kaggle/working/noorix_urdu_crnn_best.pth",map_location="cpu",weights_only=False)
torch.save(ckpt,out/"noorix_urdu_crnn_best.pth")
(out/"vocabulary.json").write_text(json.dumps({"char_to_id":char_to_id,"id_to_char":{str(k):v for k,v in id_to_char.items()}},ensure_ascii=False,indent=2),encoding="utf-8")
(out/"metadata.json").write_text(json.dumps({"project":"NOORIX-AI","prepared_by":"Muhammad Farhan",
"architecture":"CRNN-BiLSTM-CTC","script":"Urdu Unicode","image_height":HEIGHT,"max_width":MAX_WIDTH,
"validation_CER":best_cer,"input":"grayscale line image; page segmentation is not included"},ensure_ascii=False,indent=2),encoding="utf-8")
z=shutil.make_archive("/kaggle/working/NOORIX_Urdu_Line_OCR_Muhammad_Farhan","zip",out)
print("ZIP:",z)
